# Benemérita Universidad Autónoma de Puebla (BUAP)
## Facultad de Ciencias de la Computación | Licenciatura en Ingeniería en Ciencia de Datos
### Asignatura: Estadística para Ciencia de Datos (ICDA 250)
---
# Unidad 3: Estimación Estadística
## Guía Didáctica y Laboratorio Computacional para el Profesor

---

### Enfoque Pedagógico de la Unidad:
En las unidades anteriores exploramos los datos mediante estadística descriptiva (Unidad 1) y estudiamos el comportamiento de las muestras a través del Teorema Central del Límite (Unidad 2). 

En esta **Unidad 3: Estimación Estadística**, damos el paso formal hacia la **Inferencia Estadística**. El objetivo central es responder:
> *¿Cómo podemos estimar valores desconocidos de una población completa (parámetros $\theta$) a partir de una muestra finita de datos, cuantificando de forma rigurosa nuestro nivel de incertidumbre?*

Esta guía está diseñada para que tú, como profesor, puedas impartir la clase de forma interactiva, combinando fundamentos teóricos claros, código en Python con librerías estándar de Ciencia de Datos (`scipy.stats`, `pandas`, `numpy`, `seaborn`) y aplicaciones directas sobre el conjunto de datos clínicos reales de **Accidentes Cerebrovasculares (`healthcare_stroke_data.csv`)**.

### Mapa Temático Oficial:
1. **3.1 Estimadores puntuales**: Media, proporción, varianza y mediana.
2. **3.2 Propiedades de los estimadores**: Insesgadez, consistencia, eficiencia y suficiencia.
3. **3.3 Intervalos de confianza (Fundamentos)**: Desmitificando el 95% de confianza con simulación Monte Carlo.
4. **3.4 Estimación para una población**: Intervalo $t$ de Student para medias y Wilson/Wald para proporciones.
5. **3.5 Estimación para dos poblaciones**: Comparativa de grupos (prueba $t$ de Welch y diferencia de proporciones).
6. **3.6 Introducción al método de Máxima Verosimilitud (MLE)**: La función de log-verosimilitud y optimización numérica.
7. **3.7 Aplicaciones y Dashboard Clínico**: Caso de estudio integrador en Ciencia de Datos en Salud.


## 1. Configuración del Entorno y Carga del Dataset

Cargamos el ecosistema científico estándar de Python y preparamos el conjunto de datos de pacientes (`healthcare_stroke_data.csv`). Aplicamos las transformaciones aprendidas en la Unidad 1 (imputación de valores faltantes en `bmi` con la mediana e indización limpia) para trabajar con datos listos para inferencia estadística.


In [ ]:
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual moderna para gráficos estéticos y legibles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

# Carga y preparación del dataset clínico
df = pd.read_csv("healthcare_stroke_data.csv")
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df = df[df["gender"] != "Other"].reset_index(drop=True)

print("Entorno estadístico configurado exitosamente.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | SciPy: {scipy.__version__}")
print(f"Dimensiones del dataset de trabajo: {df.shape[0]} pacientes y {df.shape[1]} variables clínicas.")
df.head(3)


## 2. Tema 3.1: Estimadores Puntuales en Ciencia de Datos

### Concepto Fundamental: Parámetro vs. Estimador Puntual
- **Parámetro ($\theta$)**: Una constante fija del mundo real que describe a toda la población (e.g. la verdadera media de glucosa en todos los seres humanos $\mu$, o la proporción global de hipertensos $p$). Es desconocido.
- **Estimador Puntual ($\hat{\theta}$)**: Una regla matemática o función aplicada a los datos muestrales que arroja un único número como nuestra "mejor estimación" de $\theta$.
  - Media muestral: $\bar{X} = \frac{1}{n} \sum_{i=1}^n X_i \quad \longrightarrow \text{estima } \mu$
  - Proporción muestral: $\hat{p} = \frac{X}{n} \quad \longrightarrow \text{estima } p$
  - Mediana muestral: $\tilde{X} = \text{Percentil 50} \quad \longrightarrow \text{estima la tendencia central en distribuciones asimétricas}$
  - Varianza muestral insesgada: $S^2 = \frac{1}{n-1} \sum_{i=1}^n (X_i - \bar{X})^2 \quad \longrightarrow \text{estima } \sigma^2$

---
### Ejercicio Práctico 1: Estimación de Indicadores Clínicos Clave
Calculamos los estimadores puntuales para las variables continuas y categóricas del estudio.


In [ ]:
# 1. Estimación puntual de la media y mediana de edad
media_edad = df["age"].mean()
mediana_edad = df["age"].median()

# 2. Estimación puntual de la glucosa promedio (afectada por asimetría)
media_glucosa = df["avg_glucose_level"].mean()
mediana_glucosa = df["avg_glucose_level"].median()

# 3. Estimación puntual de la prevalencia de ACV (proporción muestral p_hat)
n_total = len(df)
casos_stroke = df["stroke"].sum()
p_hat_stroke = casos_stroke / n_total

# 4. Estimación puntual de prevalencia de hipertensión
p_hat_hipertension = df["hypertension"].mean()

print("=" * 60)
print("           ESTIMACIONES PUNTUALES CLÍNICAS")
print("=" * 60)
print(f"Edad Promedio (x_barra):           {media_edad:.2f} años  (Mediana: {mediana_edad:.1f} años)")
print(f"Glucosa Promedio (x_barra):        {media_glucosa:.2f} mg/dL (Mediana: {mediana_glucosa:.2f} mg/dL)")
print(f"Prevalencia Muestral de ACV (p_hat): {p_hat_stroke * 100:.3f}% ({casos_stroke} de {n_total} pacientes)")
print(f"Prevalencia de Hipertensión (p_hat): {p_hat_hipertension * 100:.2f}%")
print("=" * 60)

# Reflexión didáctica para clase:
# ¿Por qué para la glucosa la media (106.15) es bastante más alta que la mediana (91.88)?
# R: Debido al sesgo positivo causado por el ~12% de pacientes con diabetes descompensada.


## 3. Tema 3.2: Propiedades Deseables de los Estimadores

¿Cómo sabemos si un estimador es "bueno" para utilizar en Ciencia de Datos? La teoría estadística formal define cuatro propiedades esenciales:

1. **Insesgadez (Unbiasedness)**: El valor esperado del estimador coincide con el parámetro poblacional:
   $$\mathbb{E}[\hat{\theta}] = \theta \quad \iff \quad \text{Sesgo}(\hat{\theta}) = \mathbb{E}[\hat{\theta}] - \theta = 0$$
   *Ejemplo*: $\bar{X}$ es insesgado para $\mu$. En contraste, la varianza con divisor $n$ es sesgada, por lo que usamos $n-1$ (corrección de Bessel).
2. **Eficiencia (Efficiency)**: Entre dos estimadores insesgados, el más eficiente es aquel que tiene **menor varianza**:
   $$\text{Var}(\hat{\theta}_1) < \text{Var}(\hat{\theta}_2) \implies \hat{\theta}_1 \text{ es más eficiente}$$
3. **Consistencia (Consistency)**: A medida que el tamaño de muestra crece ($n \to \infty$), el estimador converge en probabilidad al parámetro real:
   $$\lim_{n \to \infty} P(|\hat{\theta}_n - \theta| < \epsilon) = 1$$
4. **Suficiencia (Sufficiency)**: El estimador extrae y resume toda la información que los datos contienen sobre el parámetro.

---
### Ejercicio Práctico 2: Demostración Visual de la Consistencia
Comprobaremos computacionalmente cómo la media muestral $\bar{X}_n$ converge hacia la media real conforme aumentamos el tamaño muestral de $n=10$ hasta $n=5,000$.


In [ ]:
np.random.seed(42)

# Consideramos la edad de todos los pacientes como nuestra población de referencia
poblacion_edad = df["age"].values
mu_verdadera = poblacion_edad.mean()

tamanos_muestra = [10, 25, 50, 100, 250, 500, 1000, 2500, 5000]
estimaciones_por_n = []
errores_por_n = []

for n_i in tamanos_muestra:
    # Extraemos 50 repeticiones para cada tamaño de muestra para ver la dispersión
    medias_rep = [np.random.choice(poblacion_edad, size=n_i, replace=True).mean() for _ in range(50)]
    estimaciones_por_n.append(medias_rep)
    errores_por_n.append(np.mean(np.abs(np.array(medias_rep) - mu_verdadera)))

# Gráfico de convergencia (Ley de los Grandes Números y Consistencia)
fig, ax = plt.subplots(figsize=(10, 5))
for idx, n_i in enumerate(tamanos_muestra):
    y_vals = estimaciones_por_n[idx]
    x_vals = [n_i] * len(y_vals)
    ax.scatter(x_vals, y_vals, color="#2b5c8f", alpha=0.35, edgecolors="none", s=25)

medias_promedio = [np.mean(vals) for vals in estimaciones_por_n]
ax.plot(tamanos_muestra, medias_promedio, color="darkorange", marker="o", linewidth=2.5, label="Media de Estimaciones")
ax.axhline(mu_verdadera, color="crimson", linestyle="--", linewidth=2, label=f"Parámetro Real μ = {mu_verdadera:.2f}")

ax.set_xscale("log")
ax.set_title("Demostración Empírica de la Consistencia de la Media Muestral (x̄ → μ)", fontweight="bold")
ax.set_xlabel("Tamaño de Muestra (n) - Escala Logarítmica")
ax.set_ylabel("Estimación Puntual de la Media (años)")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

print(f"Error Absoluto Medio con n=10:   {errores_por_n[0]:.2f} años")
print(f"Error Absoluto Medio con n=5000: {errores_por_n[-1]:.2f} años (¡Convergencia evidente!)")


## 4. Tema 3.3: Fundamentos de los Intervalos de Confianza

Una estimación puntual $\hat{\theta}$ da una sola cifra, pero **no informa sobre su margen de incertidumbre**. 

Un **Intervalo de Confianza (IC)** al nivel $(1 - \alpha)$ proporciona un rango plausible de valores para el parámetro desconocido:
$$\text{Intervalo de Confianza} = \text{Estimador Puntual} \pm (\text{Valor Crítico} \times \text{Error Estándar})$$
$$\text{IC}_{(1-\alpha)} = \hat{\theta} \pm \text{Margen de Error}$$

Donde:
- $\alpha$: Nivel de significancia (habitualmente $\alpha = 0.05 \implies \text{Confianza del } 95\%$).
- $\text{Valor Crítico}$: Factor de escala probabilístico ($Z_{\alpha/2}$ de la Normal o $t_{\alpha/2}$ de Student).
- $\text{Error Estándar } (SE)$: Desviación estándar de la distribución muestral del estimador.

---
### ¡Alerta Didáctica!: La Falacia de Interpretación más Frecuente
> ❌ **Interpretación Incorrecta**: *"Hay un 95% de probabilidad de que la verdadera media poblacional $\mu$ esté dentro del intervalo [104.9, 107.4]"*.
> 
> *¿Por qué es falso?* Porque el parámetro $\mu$ **no es una variable aleatoria**; es una constante fija del mundo real. Ya cayó o no cayó dentro del intervalo.
>
>  **Interpretación Correcta (Frecuentista)**: *"Si repitiéramos el proceso de muestreo exactamente 100 veces y construyéramos 100 intervalos de confianza al 95%, esperaríamos que aproximadamente 95 de ellos capturen al verdadero parámetro $\mu$ y alrededor de 5 fallen"*.

---
### Ejercicio Práctico 3: Simulación Gráfica de 100 Intervalos de Confianza
Generamos 100 muestras de tamaño $n=50$ y graficamos sus intervalos al 95%. Pintamos en **azul/verde** los que capturan la media real $\mu$ y en **rojo** los que fallan.


In [ ]:
np.random.seed(101)

poblacion_glucosa = df["avg_glucose_level"].values
mu_real_glu = poblacion_glucosa.mean()

num_intervalos = 100
n_muestra = 50
nivel_confianza = 0.95
alfa = 1 - nivel_confianza

intervalos = []
captura_parametro = []

for _ in range(num_intervalos):
    muestra = np.random.choice(poblacion_glucosa, size=n_muestra, replace=True)
    m = muestra.mean()
    se = muestra.std(ddof=1) / np.sqrt(n_muestra)
    # Valor crítico t de Student para n=50 (df = 49)
    t_crit = stats.t.ppf(1 - alfa/2, df=n_muestra - 1)
    margen = t_crit * se
    lim_inf = m - margen
    lim_sup = m + margen
    
    intervalos.append((m, lim_inf, lim_sup))
    # Comprobamos si el parámetro real está dentro del intervalo
    captura_parametro.append(lim_inf <= mu_real_glu <= lim_sup)

# Visualización pedagógica de los 100 intervalos
fig, ax = plt.subplots(figsize=(12, 6))

total_exitos = sum(captura_parametro)
total_fallos = num_intervalos - total_exitos

for idx, (m, inf, sup) in enumerate(intervalos):
    exito = captura_parametro[idx]
    color = "#2b5c8f" if exito else "crimson"
    alpha_line = 0.55 if exito else 1.0
    lw = 1.2 if exito else 2.5
    
    ax.plot([inf, sup], [idx, idx], color=color, alpha=alpha_line, linewidth=lw)
    ax.scatter([m], [idx], color=color, s=12 if exito else 25)

ax.axvline(mu_real_glu, color="black", linestyle="--", linewidth=2, label=f"Parámetro Real μ = {mu_real_glu:.2f}")
ax.set_title(f"Simulación de {num_intervalos} Intervalos de Confianza al 95% (Éxitos: {total_exitos} | Fallos: {total_fallos})", fontweight="bold")
ax.set_xlabel("Glucosa Promedio (mg/dL)")
ax.set_ylabel("Número de Muestra Simulada")
ax.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

print(f"Porcentaje empírico de cobertura: {total_exitos}% (¡Casi exactamente el 95% teórico esperado!)")


## 5. Tema 3.4: Estimación para una Población

En aplicaciones prácticas de Ciencia de Datos nos encontramos típicamente con dos tipos de problemas de una muestra:

### 1. Intervalo para la Media Poblacional ($\mu$) con Varianza Desconocida
Cuando no conocemos la desviación estándar poblacional $\sigma$ (lo cual ocurre en el 99.9% de los casos reales), usamos la desviación estándar muestral $s$ y la distribución **$t$ de Student** con $n - 1$ grados de libertad:
$$\bar{X} \pm t_{\alpha/2, n-1} \left( \frac{s}{\sqrt{n}} \right)$$

En Python, `scipy.stats.t.interval(confidence, df, loc, scale)` calcula este rango directamente.

---
### 2. Intervalo para una Proporción Poblacional ($p$)
Para variables categóricas o binarias (e.g. tasa de conversión, presencia de ACV):
- **Aproximación Clásica de Wald**: $\hat{p} \pm Z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$ (puede dar límites $< 0$ o $> 1$ en muestras pequeñas o proporciones raras).
- **Intervalo de Wilson Score**: Ajusta el centro del intervalo y es el estándar riguroso recomendado en bioestadística y machine learning.

---
### Ejercicio Práctico 4: Estimación de Glucosa Media y Tasa de Hipertensión


In [ ]:
# -------------------------------------------------------------
# 1. Intervalo de Confianza para la Media de Glucosa (t de Student)
# -------------------------------------------------------------
n_pacientes = len(df)
media_glu = df["avg_glucose_level"].mean()
s_glu = df["avg_glucose_level"].std(ddof=1)
se_glu = s_glu / np.sqrt(n_pacientes)

# Calculamos al 95% y al 99% de confianza
ic_glu_95 = stats.t.interval(0.95, df=n_pacientes - 1, loc=media_glu, scale=se_glu)
ic_glu_99 = stats.t.interval(0.99, df=n_pacientes - 1, loc=media_glu, scale=se_glu)

print("=" * 65)
print("     INTERVALOS DE CONFIANZA PARA LA MEDIA DE GLUCOSA (mg/dL)")
print("=" * 65)
print(f"Estimación Puntual (x_barra):     {media_glu:.2f} mg/dL (SE = {se_glu:.3f})")
print(f"Intervalo del 95% (t-Student):    [{ic_glu_95[0]:.2f}, {ic_glu_95[1]:.2f}] mg/dL")
print(f"Intervalo del 99% (t-Student):    [{ic_glu_99[0]:.2f}, {ic_glu_99[1]:.2f}] mg/dL")
print("=" * 65)

# -------------------------------------------------------------
# 2. Intervalo de Confianza para la Proporción de Hipertensión (Wilson)
# -------------------------------------------------------------
casos_hip = df["hypertension"].sum()
p_hat_hip = casos_hip / n_pacientes
z_crit = stats.norm.ppf(0.975)  # 1.96 para 95% de confianza

# Cálculo analítico del método Wilson Score
denom = 1 + (z_crit**2) / n_pacientes
centro_ajustado = (p_hat_hip + (z_crit**2) / (2 * n_pacientes)) / denom
error_ajustado = z_crit * np.sqrt((p_hat_hip * (1 - p_hat_hip) + (z_crit**2) / (4 * n_pacientes)) / n_pacientes) / denom

ic_hip_inf = centro_ajustado - error_ajustado
ic_hip_sup = centro_ajustado + error_ajustado

print()
print("=" * 65)
print("     INTERVALO DE CONFIANZA PARA LA PROPORCIÓN DE HIPERTENSIÓN")
print("=" * 65)
print(f"Prevalencia Muestral (p_hat):      {p_hat_hip * 100:.2f}% ({casos_hip} pacientes)")
print(f"Intervalo del 95% (Wilson Score):  [{ic_hip_inf * 100:.2f}%, {ic_hip_sup * 100:.2f}%]")
print("=" * 65)

# Pregunta pedagógica para los alumnos:
# ¿Por qué el intervalo al 99% es más ancho que el del 95%?
# R: Para tener mayor certeza (99%) de capturar el parámetro, debemos ampliar el rango.


## 6. Tema 3.5: Estimación para Dos Poblaciones (Comparativa de Cohortes)

En Ciencia de Datos, el valor real reside en comparar dos grupos: *¿El tratamiento A supera al control B? ¿Los usuarios premium consumen más que los estándar? ¿Los pacientes con ACV son significativamente mayores que los sanos?*

### 1. Diferencia de Medias Independientes ($\mu_1 - \mu_2$) con Varianzas Desiguales (Welch)
No debemos asumir que ambos grupos tienen varianzas idénticas. El **método de Welch** calcula los grados de libertad mediante la aproximación de Satterthwaite:
$$\text{SE}_{\text{diff}} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$
$$\text{IC} = (\bar{X}_1 - \bar{X}_2) \pm t_{\alpha/2, \nu} \times \text{SE}_{\text{diff}}$$

>  **Regla de Oro en Pruebas de Hipótesis e Inferencia**:
> Si el intervalo de confianza para $(\mu_1 - \mu_2)$ **NO incluye al cero ($0$)**, podemos afirmar con $(1-\alpha)$ de confianza que existe una **diferencia estadísticamente significativa** entre las dos poblaciones.

---
### 2. Diferencia de Proporciones ($p_1 - p_2$)
Comparamos tasas entre dos cohortes binarias:
$$\text{SE}_{(p_1 - p_2)} = \sqrt{\frac{\hat{p}_1(1-\hat{p}_1)}{n_1} + \frac{\hat{p}_2(1-\hat{p}_2)}{n_2}}$$

---
### Ejercicio Práctico 5: Comparación de Edad y Glucosa según Ocurrencia de ACV


In [ ]:
# Separamos las dos cohortes de pacientes
cohorte_stroke = df[df["stroke"] == 1]
cohorte_sanos = df[df["stroke"] == 0]

# -------------------------------------------------------------
# 1. Diferencia de Medias de EDAD: Pacientes con ACV vs Pacientes Sanos
# -------------------------------------------------------------
n1, n2 = len(cohorte_stroke), len(cohorte_sanos)
m1, m2 = cohorte_stroke["age"].mean(), cohorte_sanos["age"].mean()
s1, s2 = cohorte_stroke["age"].std(ddof=1), cohorte_sanos["age"].std(ddof=1)

diff_edad = m1 - m2
se_diff_edad = np.sqrt((s1**2 / n1) + (s2**2 / n2))

# Grados de libertad de Welch-Satterthwaite
gl_welch = ((s1**2 / n1) + (s2**2 / n2))**2 / (
    ((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1))
)
ic_diff_edad = stats.t.interval(0.95, df=gl_welch, loc=diff_edad, scale=se_diff_edad)

# -------------------------------------------------------------
# 2. Diferencia de Proporciones de HIPERTENSIÓN: Urbano vs Rural
# -------------------------------------------------------------
df_urbano = df[df["Residence_type"] == "Urban"]
df_rural = df[df["Residence_type"] == "Rural"]

p_urb = df_urbano["hypertension"].mean()
p_rur = df_rural["hypertension"].mean()
n_urb, n_rur = len(df_urbano), len(df_rural)

diff_prop_hip = p_urb - p_rur
se_diff_prop = np.sqrt((p_urb * (1 - p_urb) / n_urb) + (p_rur * (1 - p_rur) / n_rur))
z_val = stats.norm.ppf(0.975)
ic_diff_prop = (diff_prop_hip - z_val * se_diff_prop, diff_prop_hip + z_val * se_diff_prop)

print("=" * 70)
print("     COMPARACIÓN DE DOS POBLACIONES (INTERVALOS AL 95%)")
print("=" * 70)
print(f"Edad Media Pacientes ACV:         {m1:.2f} años (n={n1})")
print(f"Edad Media Pacientes Sanos:       {m2:.2f} años (n={n2})")
print(f"Diferencia Estimada (x1 - x2):    +{diff_edad:.2f} años")
print(f"IC 95% para Diferencia de Edad:   [{ic_diff_edad[0]:.2f}, {ic_diff_edad[1]:.2f}] años")
print(f"¿El intervalo contiene el 0?      {'SÍ (No significativa)' if ic_diff_edad[0] <= 0 <= ic_diff_edad[1] else 'NO (¡Diferencia Estadísticamente Significativa!)'}")
print("-" * 70)
print(f"Hipertensión Urbana vs Rural:     {p_urb*100:.2f}% vs {p_rur*100:.2f}% (Dif: {diff_prop_hip*100:.2f}%)")
print(f"IC 95% para Dif. Proporciones:    [{ic_diff_prop[0]*100:.2f}%, {ic_diff_prop[1]*100:.2f}%]")
print(f"¿El intervalo contiene el 0?      {'SÍ (No hay evidencia de diferencia por residencia)' if ic_diff_prop[0] <= 0 <= ic_diff_prop[1] else 'NO'}")
print("=" * 70)

# Gráfico comparativo de barras con intervalos de confianza al 95%
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=df, x="stroke", y="age", hue="stroke", palette=["#2b5c8f", "#d95f02"], legend=False, capsize=0.1, err_kws={"linewidth": 2}, ax=ax)
ax.set_title("Comparativa de Edad con Barras de Error (IC 95% t-Welch)", fontweight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Sanos (stroke=0)", "Con ACV (stroke=1)"])
ax.set_ylabel("Edad Media (años)")
plt.tight_layout()
plt.show()


## 7. Tema 3.6: Introducción al Método de Máxima Verosimilitud (MLE)

El método de **Máxima Verosimilitud** (*Maximum Likelihood Estimation - MLE*) es la técnica de estimación más utilizada en el Machine Learning contemporáneo (es la base del entrenamiento de Regresión Logística, Redes Neuronales y Modelos Probabilísticos).

### Intuición Sencilla:
Imaginemos que observamos un conjunto de datos $\{x_1, x_2, \dots, x_n\}$. La pregunta fundamental de MLE es:
> *¿Cuáles parámetros $\theta$ hacen que los datos que acabamos de observar tengan la **mayor probabilidad matemática** de haber ocurrido?*

1. **Función de Verosimilitud ($L(\theta)$)**:
   $$L(\theta) = f(x_1; \theta) \cdot f(x_2; \theta) \cdots f(x_n; \theta) = \prod_{i=1}^n f(x_i; \theta)$$
2. **Función de Log-Verosimilitud ($\ell(\theta)$)**:
   Como el producto de miles de probabilidades pequeñas provoca desbordamiento (*underflow* numérico), aplicamos logaritmo natural, transformando productos en sumas:
   $$\ell(\theta) = \ln L(\theta) = \sum_{i=1}^n \ln f(x_i; \theta)$$
3. **Optimización**: Buscamos el valor $\hat{\theta}_{\text{MLE}}$ donde la curva de log-verosimilitud alcanza su punto más alto (máximo global).

---
### Ejercicio Práctico 6: Curva de Log-Verosimilitud y Ajuste con SciPy


In [ ]:
# 1. Ajuste analítico de una Normal por Máxima Verosimilitud usando SciPy
datos_edad = df["age"].values
mu_mle, sigma_mle = stats.norm.fit(datos_edad)

print("=" * 60)
print("     ESTIMACIÓN POR MÁXIMA VEROSIMILITUD (MLE) CON SCIPY")
print("=" * 60)
print(f"Parámetro mu_MLE estimado:    {mu_mle:.2f} años")
print(f"Parámetro sigma_MLE estimado: {sigma_mle:.2f} años")
print("=" * 60)

# 2. Visualización didáctica de la curva de Log-Verosimilitud
# Evaluamos la log-verosimilitud para diferentes valores de prueba de mu
valores_mu_prueba = np.linspace(35, 52, 200)
log_verosimilitudes = []

for mu_test in valores_mu_prueba:
    # Suma de los logaritmos de la densidad normal: sum(ln(pdf))
    log_l = np.sum(stats.norm.logpdf(datos_edad, loc=mu_test, scale=sigma_mle))
    log_verosimilitudes.append(log_l)

log_verosimilitudes = np.array(log_verosimilitudes)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(valores_mu_prueba, log_verosimilitudes, color="#2b5c8f", linewidth=2.5, label="Log-Verosimilitud l(mu)")
ax.axvline(mu_mle, color="crimson", linestyle="--", linewidth=2, label=f"Máximo MLE en mu = {mu_mle:.2f}")

ax.scatter([mu_mle], [np.max(log_verosimilitudes)], color="crimson", s=70, zorder=5)
ax.annotate(f"Cima de la Montaña (Pico)\nmu_MLE = {mu_mle:.2f}", 
            xy=(mu_mle, np.max(log_verosimilitudes)), 
            xytext=(mu_mle + 1.5, np.max(log_verosimilitudes) - 1500),
            arrowprops=dict(facecolor="black", shrink=0.08, width=1.5, headwidth=8),
            fontsize=10, fontweight="bold")

ax.set_title("Curva de Log-Verosimilitud: Búsqueda del Parámetro Óptimo (MLE)", fontweight="bold")
ax.set_xlabel("Valor de Prueba del Parámetro Media (mu)")
ax.set_ylabel("Log-Verosimilitud l(mu)")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


## 8. Tema 3.7: Aplicaciones y Caso de Estudio en Ciencia de Datos

Como actividad de consolidación, generamos un **Healthcare Confidence Dashboard** que integra en una sola vista ejecutiva los intervalos de confianza clínicos y el ajuste de distribuciones para la toma de decisiones basada en datos.


In [ ]:
# Generación del Dashboard Ejecutivo de la Unidad 3
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Subplot 1: Estimación de Medias con IC 95% (Glucosa por Presencia de Cardiopatía)
sns.pointplot(data=df, x="heart_disease", y="avg_glucose_level", color="#2b5c8f", capsize=0.15, ax=axes[0])
axes[0].set_title("1. Media de Glucosa con IC 95%", fontweight="bold")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Sin Cardiopatía", "Con Cardiopatía"])
axes[0].set_ylabel("Glucosa Promedio (mg/dL)")

# Subplot 2: Intervalos de Confianza para Proporciones de Factores de Riesgo
factores = ["hypertension", "heart_disease", "stroke"]
nombres = ["Hipertensión", "Cardiopatía", "ACV (Stroke)"]
proporciones = [df[f].mean() * 100 for f in factores]
errores_prop = [1.96 * np.sqrt((p/100)*(1 - p/100)/len(df)) * 100 for p in proporciones]

axes[1].errorbar(x=nombres, y=proporciones, yerr=errores_prop, fmt="o", color="#2b5c8f", ecolor="crimson", elinewidth=2.5, capsize=6)
axes[1].set_title("2. Prevalencias Poblacionales con IC 95%", fontweight="bold")
axes[1].set_ylabel("Prevalencia Estimada (%)")

# Subplot 3: Ajuste de Densidad Normal MLE vs Histograma Real de Edad
x_grid = np.linspace(df["age"].min(), df["age"].max(), 200)
pdf_mle = stats.norm.pdf(x_grid, loc=mu_mle, scale=sigma_mle)

sns.histplot(df["age"], stat="density", bins=25, color="#41b6c4", alpha=0.45, ax=axes[2])
axes[2].plot(x_grid, pdf_mle, color="darkred", linewidth=2.5, label="Ajuste Normal MLE")
axes[2].set_title("3. Distribución de Edad y Modelo MLE", fontweight="bold")
axes[2].set_xlabel("Edad (años)")
axes[2].legend()

plt.suptitle("Dashboard de Inferencia y Estimación Estadística en Pacientes", fontsize=14, fontweight="bold")
plt.subplots_adjust(top=0.92, hspace=0.35, wspace=0.25)
plt.show()


## 9. Guía Didáctica y Orientaciones Pedagógicas para el Docente

Esta sección final está diseñada para orientar tu exposición en clase y estructurar la discusión con tus alumnos:

---

### Preguntas Socráticas para Detonar el Pensamiento Crítico:
1. **Sobre la Interpretación de Intervalos (Tema 3.3)**:
   - *Pregunta a la clase*: "Si calculamos que el intervalo de edad media es [42.6, 43.8] años al 95%, ¿es correcto decir que un paciente elegido al azar tiene un 95% de probabilidad de tener entre 42 y 43 años?"
   - *Guía de respuesta*: No. El intervalo de confianza estima el **parámetro promedio poblacional $\mu$**, no el valor de una observación individual $X_i$. Confundir un intervalo de confianza con un intervalo de predicción o dispersión es un error común.
2. **Sobre el Ancho del Intervalo y Tamaño de Muestra (Tema 3.4)**:
   - *Pregunta*: "¿Qué pasaría si en lugar de 5,109 pacientes solo tuviéramos 50? ¿El intervalo sería más ancho o más estrecho?"
   - *Guía de respuesta*: Sería mucho más ancho, porque el error estándar $SE = s/\sqrt{n}$ aumenta al disminuir $n$. A menor tamaño de muestra, mayor incertidumbre.
3. **Sobre la Comparativa de Dos Poblaciones (Tema 3.5)**:
   - *Pregunta*: "Observamos que la diferencia de edad entre pacientes con ACV y sanos tiene un intervalo de [24.05, 27.46] años. ¿Qué nos indica que ambos límites sean positivos y no incluyan el 0?"
   - *Guía de respuesta*: Indica que con un 95% de confianza, los pacientes con ACV son al menos 24 años mayores en promedio, descartando el azar como explicación.
4. **Sobre Máxima Verosimilitud (Tema 3.6)**:
   - *Pregunta*: "¿Cuál es la diferencia conceptual entre la función de probabilidad y la función de verosimilitud?"
   - *Guía de respuesta*: En la probabilidad, los parámetros son fijos y evaluamos los datos; en la verosimilitud, los datos son los que están fijos (lo que observamos) y evaluamos los parámetros.

---
### Conexión con la Siguiente Unidad (Unidad 4: Pruebas de Hipótesis):
Señala a tus alumnos que la estimación por intervalos de confianza es la otra cara de la moneda de las **Pruebas de Hipótesis**. Cualquier valor que quede **fuera** de un intervalo de confianza del 95% sería **rechazado** en una prueba de hipótesis bilateral con nivel de significancia $\alpha = 0.05$.
